In [1]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timezone

In [2]:
now_utc = datetime.now(timezone.utc)
spx = yf.Ticker("^SPX")

expiries = ['2026-12-18','2027-01-15'] # consistent expiry dates. these are the next two monthly expiries

available = set(spx.options) # list of all the option dates.
for e in expiries: #check if the expiries are currently listed
    if e not in available:
        print("Expiry not listed: ", e)


spot = yf.Ticker("^SPX").history(period="1d")["Close"].iloc[-1] #current spot price of spx
frames = []

for exp in expiries: #iterate over the expiry dates we are going to track
    ch = spx.option_chain(exp) #option chain
    for df, kind in [(ch.calls, "C"), (ch.puts, "P")]: 
        df = df.copy()
        df["expiry"], df["type"] = exp, kind
        frames.append(df) # get all the dfs for each of the expiries/option kind

snap = pd.concat(frames) # concatenate them all together
snap["snapshot_utc"], snap["spot"] = now_utc, spot # add our snapshot time and the spot price
snap.to_csv(f"spx-{now_utc:%m_%d_%Y-%H%M}.csv", index=False) #save the csv